# Group Members: Jonah Blackmon, Emily Lucas, Kevin Le
# Eids: jkb3578, 

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np
import re

%matplotlib inline

pd.__version__

 

'3.1.1'

In [3]:
def age_to_years(age):
    if pd.isna(age):
        return np.nan

    age = age.lower().strip()

    match = re.match(r"(\d+(?:\.\d+)?)\s*(day|days|week|weeks|month|months|year|years)", age)

    if not match:
        return np.nan

    value = float(match.group(1))
    unit = match.group(2)

    conversions = {
        "day": 1 / 365.25,
        "days": 1 / 365.25,
        "week": 7 / 365.25,
        "weeks": 7 / 365.25,
        "month": 1 / 12,
        "months": 1 / 12,
        "year": 1,
        "years": 1
    }

    return value * conversions[unit]

In [4]:
outcome_df = pd.read_csv (filepath_or_buffer="./adoption_data_outcome.csv")
intake_df = pd.read_csv (filepath_or_buffer="./adoption_data_intake.csv")

duplicate_cols = intake_df.columns.intersection(outcome_df.columns).difference(["Animal ID"])

intake_df = intake_df.drop(columns=duplicate_cols)

# Merge intake and outcome csv together, keeping only animals that exist in both datasets
df = pd.merge(outcome_df, intake_df, on="Animal ID", how="inner")
display(df.head())

,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color,Found Location,Intake Type,Intake Condition,Sex upon Intake,Age upon Intake
0,A668305,2012-12-01,NaN,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow,10101 W Parmer Ln in Austin (TX),Wildlife,Injured,Unknown,1 year
1,A673335,2012-02-22,NaN,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray,4415 Secluded Hollow in Austin (TX),Wildlife,Sick,Unknown,2 years
2,A675999,2013-04-03,NaN,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green,Levander Loop in Austin (TX),Wildlife,Injured,Unknown,1 year
3,A679066,2014-04-16,NaN,2014-05-16T00:00:00-05:00,05-2014,NaN,NaN,Other,Unknown,4 weeks,Rabbit Sh,Brown,Austin (TX),Wildlife,Normal,Unknown,4 weeks
4,A680855,2014-05-25,NaN,2014-06-10T00:00:00-05:00,06-2014,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black,2000 N. Ih35 in Austin (TX),Stray,Normal,Unknown,2 weeks


In [5]:
# Unecessary columns
df = df.drop(columns=["Animal ID", "DateTime", "MonthYear", "Outcome Subtype", "Date of Birth", "Found Location", "Color", "Age upon Outcome"])
df = df[~((df["Breed"] == "Unknown") &(df["Animal Type"] == "Other"))]
display(df.head())

,Name,Outcome Type,Animal Type,Sex upon Outcome,Breed,Intake Type,Intake Condition,Sex upon Intake,Age upon Intake
0,NaN,Transfer,Other,Unknown,Turtle Mix,Wildlife,Injured,Unknown,1 year
1,NaN,Euthanasia,Other,Unknown,Raccoon,Wildlife,Sick,Unknown,2 years
2,NaN,Transfer,Other,Unknown,Turtle Mix,Wildlife,Injured,Unknown,1 year
3,NaN,NaN,Other,Unknown,Rabbit Sh,Wildlife,Normal,Unknown,4 weeks
4,NaN,Transfer,Bird,Unknown,Duck,Stray,Normal,Unknown,2 weeks


In [6]:
# Convert "Name" to a boolean for existence
df["Name"] = df["Name"].notna().astype(int)
display(df.head())

,Name,Outcome Type,Animal Type,Sex upon Outcome,Breed,Intake Type,Intake Condition,Sex upon Intake,Age upon Intake
0,0,Transfer,Other,Unknown,Turtle Mix,Wildlife,Injured,Unknown,1 year
1,0,Euthanasia,Other,Unknown,Raccoon,Wildlife,Sick,Unknown,2 years
2,0,Transfer,Other,Unknown,Turtle Mix,Wildlife,Injured,Unknown,1 year
3,0,NaN,Other,Unknown,Rabbit Sh,Wildlife,Normal,Unknown,4 weeks
4,0,Transfer,Bird,Unknown,Duck,Stray,Normal,Unknown,2 weeks


In [7]:
# Multi-label encode breed, can be more than one
df["Breed"] = df["Breed"].str.replace(" Mix", "", regex=False)

# Drop NaN outcomes, can't predict that
# df = df.dropna()

breed_encoded = df["Breed"].str.get_dummies(sep="/")
breed_encoded = breed_encoded.add_prefix("Breed_")
df = pd.concat([df.drop(columns=["Breed"]), breed_encoded], axis=1)
display(df.head())

,Name,Outcome Type,Animal Type,Sex upon Outcome,Intake Type,Intake Condition,Sex upon Intake,Age upon Intake,Breed_Abyssinian,Breed_Affenpinscher,...,Breed_Whimbrel,Breed_Whippet,Breed_Wire Hair Fox Terrier,Breed_Wirehaired Pointing Griffon,Breed_Wirehaired Vizsla,Breed_Wolf Hybrid,Breed_Wren,Breed_Wyandottes,Breed_Yorkshire,Breed_Yorkshire Terrier
0,0,Transfer,Other,Unknown,Wildlife,Injured,Unknown,1 year,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,Euthanasia,Other,Unknown,Wildlife,Sick,Unknown,2 years,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,Transfer,Other,Unknown,Wildlife,Injured,Unknown,1 year,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,NaN,Other,Unknown,Wildlife,Normal,Unknown,4 weeks,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,Transfer,Bird,Unknown,Stray,Normal,Unknown,2 weeks,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
df["Age upon Intake"] = df["Age upon Intake"].apply(age_to_years)
display(df.head())

,Name,Outcome Type,Animal Type,Sex upon Outcome,Intake Type,Intake Condition,Sex upon Intake,Age upon Intake,Breed_Abyssinian,Breed_Affenpinscher,...,Breed_Whimbrel,Breed_Whippet,Breed_Wire Hair Fox Terrier,Breed_Wirehaired Pointing Griffon,Breed_Wirehaired Vizsla,Breed_Wolf Hybrid,Breed_Wren,Breed_Wyandottes,Breed_Yorkshire,Breed_Yorkshire Terrier
0,0,Transfer,Other,Unknown,Wildlife,Injured,Unknown,1.00000,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,Euthanasia,Other,Unknown,Wildlife,Sick,Unknown,2.00000,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,Transfer,Other,Unknown,Wildlife,Injured,Unknown,1.00000,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,NaN,Other,Unknown,Wildlife,Normal,Unknown,0.07666,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,Transfer,Bird,Unknown,Stray,Normal,Unknown,0.03833,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
df["Sex"] = df["Sex upon Intake"].str.extract(r"(Male|Female)", flags=re.IGNORECASE)
df["Sex"] = df["Sex"].str.capitalize().fillna("Unknown")

df["Reproductive Status"] = "Unknown"

df.loc[df["Sex upon Intake"].str.contains(
    "Neutered", case=False, na=False
), "Reproductive Status"] = "Neutered"

df.loc[df["Sex upon Intake"].str.contains(
    "Spayed", case=False, na=False
), "Reproductive Status"] = "Spayed"

df.loc[df["Sex upon Intake"].str.contains(
    "Intact", case=False, na=False
), "Reproductive Status"] = "Intact"

df = df.drop(columns=["Sex upon Intake", "Sex upon Outcome"])
display(df.head())

,Name,Outcome Type,Animal Type,Intake Type,Intake Condition,Age upon Intake,Breed_Abyssinian,Breed_Affenpinscher,Breed_Afghan Hound,Breed_African,...,Breed_Wire Hair Fox Terrier,Breed_Wirehaired Pointing Griffon,Breed_Wirehaired Vizsla,Breed_Wolf Hybrid,Breed_Wren,Breed_Wyandottes,Breed_Yorkshire,Breed_Yorkshire Terrier,Sex,Reproductive Status
0,0,Transfer,Other,Wildlife,Injured,1.00000,0,0,0,0,...,0,0,0,0,0,0,0,0,Unknown,Unknown
1,0,Euthanasia,Other,Wildlife,Sick,2.00000,0,0,0,0,...,0,0,0,0,0,0,0,0,Unknown,Unknown
2,0,Transfer,Other,Wildlife,Injured,1.00000,0,0,0,0,...,0,0,0,0,0,0,0,0,Unknown,Unknown
3,0,NaN,Other,Wildlife,Normal,0.07666,0,0,0,0,...,0,0,0,0,0,0,0,0,Unknown,Unknown
4,0,Transfer,Bird,Stray,Normal,0.03833,0,0,0,0,...,0,0,0,0,0,0,0,0,Unknown,Unknown


In [10]:
animal_encoded = pd.get_dummies(df["Animal Type"], prefix="Animal", dtype=int)
df = pd.concat([df.drop(columns=["Animal Type"]), animal_encoded], axis=1)

categorical_cols = ["Intake Type", "Intake Condition", "Sex", "Reproductive Status"]
df = pd.get_dummies(df, columns=categorical_cols, dtype=int)

display(df.head())

,Name,Outcome Type,Age upon Intake,Breed_Abyssinian,Breed_Affenpinscher,Breed_Afghan Hound,Breed_African,Breed_Airedale Terrier,Breed_Akbash,Breed_Akita,...,Intake Condition_Sick,Intake Condition_Space,Intake Condition_Unknown,Sex_Female,Sex_Male,Sex_Unknown,Reproductive Status_Intact,Reproductive Status_Neutered,Reproductive Status_Spayed,Reproductive Status_Unknown
0,0,Transfer,1.00000,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1
1,0,Euthanasia,2.00000,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,1
2,0,Transfer,1.00000,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1
3,0,NaN,0.07666,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1
4,0,Transfer,0.03833,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1


In [11]:
df = df.drop(columns=["Breed_Unknown", "Animal_Other"])
display(df.head())

,Name,Outcome Type,Age upon Intake,Breed_Abyssinian,Breed_Affenpinscher,Breed_Afghan Hound,Breed_African,Breed_Airedale Terrier,Breed_Akbash,Breed_Akita,...,Intake Condition_Sick,Intake Condition_Space,Intake Condition_Unknown,Sex_Female,Sex_Male,Sex_Unknown,Reproductive Status_Intact,Reproductive Status_Neutered,Reproductive Status_Spayed,Reproductive Status_Unknown
0,0,Transfer,1.00000,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1
1,0,Euthanasia,2.00000,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,1
2,0,Transfer,1.00000,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1
3,0,NaN,0.07666,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1
4,0,Transfer,0.03833,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1


In [14]:
# Move outcome type to the last column
outcome = df.pop("Outcome Type")
df["Outcome Type"] = outcome
df = df.dropna(subset="Outcome Type")
display(df.head())

/var/folders/jv/pp3qtc5s4px1b2vtnzrvfrlc0000gn/T/ipykernel_41851/1938821127.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["Outcome Type"] = outcome


,Name,Age upon Intake,Breed_Abyssinian,Breed_Affenpinscher,Breed_Afghan Hound,Breed_African,Breed_Airedale Terrier,Breed_Akbash,Breed_Akita,Breed_Alaskan Husky,...,Intake Condition_Space,Intake Condition_Unknown,Sex_Female,Sex_Male,Sex_Unknown,Reproductive Status_Intact,Reproductive Status_Neutered,Reproductive Status_Spayed,Reproductive Status_Unknown,Outcome Type
0,0,1.00000,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,Transfer
1,0,2.00000,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,Euthanasia
2,0,1.00000,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,Transfer
4,0,0.03833,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,Transfer
5,0,0.03833,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,Transfer


(221178, 433)